In [ ]:
import io
import re
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from google.colab import files

In [ ]:
# Change this if needed
CATCHMENT_AREA_M2 = 333_700_000  # Wansbeck

# Default baseline scenario for % change calculations
BASELINE_SCENARIO_ID = "R1-T70"

# Totals we want to extract from the Excel exports
TOTAL_ROW_LABELS = {
    "Number of opportunities": "Total Outputs",
    "Storage": "Total Storage (m³)",
    "Area": "Total Area (m²)",
    "Net carbon sequestration": "Total Net Carbon",
    "Net habitat units": "Total Net Habitat",
    "Cost": "Total Cost (£)",
}

# Final output columns
OUTPUT_COLUMNS = [
    "Scenario ID",
    "Rank",
    "Threshold",
    "Ordered By",
    "Runoff Attenuation Feature Outputs",
    "Floodplain Reconnection Outputs",
    "Large Woody Debris Outputs",
    "Tree Planting Outputs",
    "Wet Woodland Outputs",
    "Buffer Strip Outputs",
    "Soil Management Outputs",
    "Peat Management Outputs",
    "Gully Stuffing Outputs",
    "Grip Blocking Outputs",
    "Total Outputs",
    "Total Storage (m³)",
    "Total Area (m²)",
    "% Catchment Covered",
    "Total Net Carbon",
    "Total Net Habitat",
    "Total Cost (£)",
    "Dominant Intervention",
    "% Change Outputs (vs Baseline)",
    "% Change Storage (vs Baseline)",
    "% Change Cost (vs Baseline)",
    "File Source",
    "Notes",
]

In [ ]:
# creating a rule based interpreter
# reads the scenario ID strings

def parse_scenario_id(scenario_id: str) -> dict:
    """
    Accepts:
        OBS-R1-T70
        OBC-R1-T70
        OBH-R1-T70
        R1-T70


    Returns:
        {
            "Scenario ID": "...",
            "Rank": 1,
            "Threshold": 70,
            "Ordered By": "Storage" / "Carbon" / "Habitat" / "Default"
        }
    """
    # returns the scenario into readable format
    # then perform a cleaning process, removing spaces etc. for clean data.
    scenario_id = scenario_id.strip().upper().replace('"', "").replace("'", "")

    pattern = r"(?:(OBS|OBC|OBH)-)?R(\d+)-T(\d+)"
    # sets the structure, some may have the order by function too, but not necessarily all.
    #  captures rank and threshold too.
    match = re.fullmatch(pattern, scenario_id)

    if not match:
        raise ValueError(
            f"Scenario ID '{scenario_id}' is invalid. Use formats like "
            f"'OBS-R1-T70', 'OBC-R1-T70', 'OBH-R1-T70', or 'R1-T70'."
        )
# provides error is format isnt correct, to ensure correct inputs and consistency
    order_code, rank, threshold = match.groups()
# extrats and matches the components.
    order_map = {
        "OBS": "Storage",
        "OBC": "Carbon",
        "OBH": "Habitat",
        None: "Default",
    }

    return {
        "Scenario ID": scenario_id,
        "Rank": int(rank),
        "Threshold": int(threshold),
        "Ordered By": order_map[order_code],
    }

In [ ]:
def find_total_column(ws):
    """
    Finds the 'Total values' column dynamically from row 1.
    Falls back to column 11 if not found.
    """
    #
    headers = [ws.cell(row=1, column=col).value for col in range(1, ws.max_column + 1)]

    for i, header in enumerate(headers, start=1):
        if str(header).strip().lower() == "total values":
            return i

    print("Warning: 'Total values' column not found. Falling back to column 11 (K).")
    return 11


def find_row_by_label(ws, label: str):
    """
    Finds the row number where column A matches the given label.
    """
    for row in range(1, ws.max_row + 1):
        value = ws.cell(row=row, column=1).value
        if str(value).strip() == label:
            return row
    return None


def to_number(value):
    """
    Converts values safely to numeric where possible.
    """
    return pd.to_numeric(value, errors="coerce")

In [ ]:
def standardise_intervention_name(name: str) -> str:
    """
    Maps workbook header names to your preferred output column names.
    Handles slight naming differences safely.
    """
    if name is None:
        return None

    name = str(name).strip().lower()

    mapping = {
        "runoff attenuation feature": "Runoff Attenuation Feature Outputs",
        "runoff attenuation features": "Runoff Attenuation Feature Outputs",

        "floodplain reconnection": "Floodplain Reconnection Outputs",

        "large woody debris": "Large Woody Debris Outputs",

        "tree plant": "Tree Planting Outputs",
        "tree planting": "Tree Planting Outputs",

        "wet wood": "Wet Woodland Outputs",
        "wet woodland": "Wet Woodland Outputs",

        "buffer strip": "Buffer Strip Outputs",
        "buffer strips": "Buffer Strip Outputs",

        "soil management": "Soil Management Outputs",

        "peat management": "Peat Management Outputs",

        "gully stuffing": "Gully Stuffing Outputs",

        "grip blocking": "Grip Blocking Outputs",
    }

    return mapping.get(name)

In [ ]:
def extract_one_export(file_bytes: bytes, scenario_id: str, file_name: str) -> dict:
    scenario_meta = parse_scenario_id(scenario_id)

    wb = load_workbook(io.BytesIO(file_bytes), data_only=True)
    ws = wb[wb.sheetnames[0]]

    total_col = find_total_column(ws)

    row_data = {col: None for col in OUTPUT_COLUMNS}
    row_data.update(scenario_meta)
    row_data["File Source"] = file_name
    row_data["Notes"] = ""

    # Initialise intervention output columns to zero
    intervention_output_cols = [
        "Runoff Attenuation Feature Outputs",
        "Floodplain Reconnection Outputs",
        "Large Woody Debris Outputs",
        "Tree Planting Outputs",
        "Wet Woodland Outputs",
        "Buffer Strip Outputs",
        "Soil Management Outputs",
        "Peat Management Outputs",
        "Gully Stuffing Outputs",
        "Grip Blocking Outputs",
    ]

    for col in intervention_output_cols:
        row_data[col] = 0

    # ----- Extract totals -----
    for raw_label, output_name in TOTAL_ROW_LABELS.items():
        row_num = find_row_by_label(ws, raw_label)
        if row_num is not None:
            row_data[output_name] = to_number(ws.cell(row=row_num, column=total_col).value)

    # ----- Extract intervention counts from the "Number of opportunities" row -----
    count_row = find_row_by_label(ws, "Number of opportunities")
    if count_row is None:
        raise ValueError(f"'Number of opportunities' row not found in file: {file_name}")

    # Intervention columns are from B to the column before total_col
    for col in range(2, total_col):
        raw_header = ws.cell(row=1, column=col).value
        clean_header = standardise_intervention_name(raw_header)

        if clean_header is not None:
            value = ws.cell(row=count_row, column=col).value
            row_data[clean_header] = to_number(value)

    # ----- Calculate dominant intervention -----
    counts_for_dominance = {
        col: row_data[col]
        for col in intervention_output_cols
        if pd.notna(row_data[col])
    }

    if counts_for_dominance:
        row_data["Dominant Intervention"] = max(
            counts_for_dominance,
            key=lambda k: counts_for_dominance[k]
        ).replace(" Outputs", "")

    # ----- % catchment covered -----
    if pd.notna(row_data["Total Area (m²)"]) and CATCHMENT_AREA_M2:
        row_data["% Catchment Covered"] = (
            row_data["Total Area (m²)"] / CATCHMENT_AREA_M2 * 100
        )

    return row_data

In [ ]:
uploaded = files.upload()
uploaded_filenames = sorted(uploaded.keys())

print("Files uploaded in this order:")
for i, name in enumerate(uploaded_filenames, start=1):
    print(f"{i}. {name}")

scenario_input = input(
    "\nEnter Scenario IDs in the SAME ORDER, comma-separated.\n"
    "Example:\n"
    "OBS-R1-T70, OBC-R1-T70, OBH-R1-T70, R1-T70\n\n"
).strip()

scenario_ids = [x.strip() for x in scenario_input.split(",")]

if len(scenario_ids) != len(uploaded_filenames):
    raise ValueError(
        f"You entered {len(scenario_ids)} Scenario IDs but uploaded {len(uploaded_filenames)} files."
    )

print("\nScenario mapping:")
for file_name, scenario_id in zip(uploaded_filenames, scenario_ids):
    print(f"{file_name} --> {scenario_id}")

Saving OBS-R1-T50.xlsx to OBS-R1-T50.xlsx
Saving OBH-R1-T50.xlsx to OBH-R1-T50.xlsx
Saving OBC-R1-T50.xlsx to OBC-R1-T50.xlsx
Saving R1-T50.xlsx to R1-T50.xlsx
Files uploaded in this order:
1. OBC-R1-T50.xlsx
2. OBH-R1-T50.xlsx
3. OBS-R1-T50.xlsx
4. R1-T50.xlsx

Enter Scenario IDs in the SAME ORDER, comma-separated.
Example:
OBS-R1-T70, OBC-R1-T70, OBH-R1-T70, R1-T70

OBC-R1-T50, OBH-R1-T50, OBS-R1-T50, R1-T50

Scenario mapping:
OBC-R1-T50.xlsx --> OBC-R1-T50
OBH-R1-T50.xlsx --> OBH-R1-T50
OBS-R1-T50.xlsx --> OBS-R1-T50
R1-T50.xlsx --> R1-T50


In [ ]:
all_rows = []

for file_name, scenario_id in zip(uploaded_filenames, scenario_ids):
    print(f"Processing {file_name} as {scenario_id}")
    file_bytes = uploaded[file_name]
    row = extract_one_export(file_bytes, scenario_id, file_name)
    all_rows.append(row)

summary_df = pd.DataFrame(all_rows)

# Ensure columns are in the preferred order
summary_df = summary_df[OUTPUT_COLUMNS]

summary_df

Processing OBC-R1-T50.xlsx as OBC-R1-T50
Processing OBH-R1-T50.xlsx as OBH-R1-T50
Processing OBS-R1-T50.xlsx as OBS-R1-T50
Processing R1-T50.xlsx as R1-T50


,Scenario ID,Rank,Threshold,Ordered By,Runoff Attenuation Feature Outputs,Floodplain Reconnection Outputs,Large Woody Debris Outputs,Tree Planting Outputs,Wet Woodland Outputs,Buffer Strip Outputs,...,% Catchment Covered,Total Net Carbon,Total Net Habitat,Total Cost (£),Dominant Intervention,% Change Outputs (vs Baseline),% Change Storage (vs Baseline),% Change Cost (vs Baseline),File Source,Notes
0,OBC-R1-T50,1,50,Carbon,0,340,19,3911,671,108,...,21.142965,88544.371658,62967.032614,20582552.5,Tree Planting,None,None,None,OBC-R1-T50.xlsx,
1,OBH-R1-T50,1,50,Habitat,2418,842,48,245,554,59,...,24.408278,22336.478805,27272.761678,33463555.0,Runoff Attenuation Feature,None,None,None,OBH-R1-T50.xlsx,
2,OBS-R1-T50,1,50,Storage,2418,842,48,245,554,59,...,24.408278,22336.478805,27272.761678,33463555.0,Runoff Attenuation Feature,None,None,None,OBS-R1-T50.xlsx,
3,R1-T50,1,50,Default,1203,304,593,472,190,1129,...,25.475764,20605.846268,18022.772490,20763785.0,Runoff Attenuation Feature,None,None,None,R1-T50.xlsx,


In [ ]:
def add_baseline_comparisons(df: pd.DataFrame, baseline_id: str = BASELINE_SCENARIO_ID) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [
        "Total Outputs",
        "Total Storage (m³)",
        "Total Area (m²)",
        "% Catchment Covered",
        "Total Net Carbon",
        "Total Net Habitat",
        "Total Cost (£)",
        "Runoff Attenuation Feature Outputs",
        "Floodplain Reconnection Outputs",
        "Large Woody Debris Outputs",
        "Tree Planting Outputs",
        "Wet Woodland Outputs",
        "Buffer Strip Outputs",
        "Soil Management Outputs",
        "Peat Management Outputs",
        "Gully Stuffing Outputs",
        "Grip Blocking Outputs",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    baseline_rows = df[df["Scenario ID"].str.upper() == baseline_id.upper()]

    if baseline_rows.empty:
        print(f"Warning: baseline '{baseline_id}' not found. % change columns left blank.")
        return df

    baseline = baseline_rows.iloc[0]

    if pd.notna(baseline["Total Outputs"]) and baseline["Total Outputs"] != 0:
        df["% Change Outputs (vs Baseline)"] = (
            (df["Total Outputs"] - baseline["Total Outputs"]) / baseline["Total Outputs"] * 100
        )

    if pd.notna(baseline["Total Storage (m³)"]) and baseline["Total Storage (m³)"] != 0:
        df["% Change Storage (vs Baseline)"] = (
            (df["Total Storage (m³)"] - baseline["Total Storage (m³)"]) / baseline["Total Storage (m³)"] * 100
        )

    if pd.notna(baseline["Total Cost (£)"]) and baseline["Total Cost (£)"] != 0:
        df["% Change Cost (vs Baseline)"] = (
            (df["Total Cost (£)"] - baseline["Total Cost (£)"]) / baseline["Total Cost (£)"] * 100
        )

    return df


summary_df = add_baseline_comparisons(summary_df, baseline_id=BASELINE_SCENARIO_ID)
summary_df

,Scenario ID,Rank,Threshold,Ordered By,Runoff Attenuation Feature Outputs,Floodplain Reconnection Outputs,Large Woody Debris Outputs,Tree Planting Outputs,Wet Woodland Outputs,Buffer Strip Outputs,...,% Catchment Covered,Total Net Carbon,Total Net Habitat,Total Cost (£),Dominant Intervention,% Change Outputs (vs Baseline),% Change Storage (vs Baseline),% Change Cost (vs Baseline),File Source,Notes
0,OBC-R1-T50,1,50,Carbon,0,340,19,3911,671,108,...,21.142965,88544.371658,62967.032614,20582552.5,Tree Planting,None,None,None,OBC-R1-T50.xlsx,
1,OBH-R1-T50,1,50,Habitat,2418,842,48,245,554,59,...,24.408278,22336.478805,27272.761678,33463555.0,Runoff Attenuation Feature,None,None,None,OBH-R1-T50.xlsx,
2,OBS-R1-T50,1,50,Storage,2418,842,48,245,554,59,...,24.408278,22336.478805,27272.761678,33463555.0,Runoff Attenuation Feature,None,None,None,OBS-R1-T50.xlsx,
3,R1-T50,1,50,Default,1203,304,593,472,190,1129,...,25.475764,20605.846268,18022.772490,20763785.0,Runoff Attenuation Feature,None,None,None,R1-T50.xlsx,


In [ ]:
order_sort = {"Default": 0, "Storage": 1, "Carbon": 2, "Habitat": 3}

summary_df["Ordered By Sort"] = summary_df["Ordered By"].map(order_sort)
summary_df = summary_df.sort_values(
    by=["Rank", "Threshold", "Ordered By Sort", "Scenario ID"]
).drop(columns=["Ordered By Sort"]).reset_index(drop=True)

summary_df

,Scenario ID,Rank,Threshold,Ordered By,Runoff Attenuation Feature Outputs,Floodplain Reconnection Outputs,Large Woody Debris Outputs,Tree Planting Outputs,Wet Woodland Outputs,Buffer Strip Outputs,...,% Catchment Covered,Total Net Carbon,Total Net Habitat,Total Cost (£),Dominant Intervention,% Change Outputs (vs Baseline),% Change Storage (vs Baseline),% Change Cost (vs Baseline),File Source,Notes
0,R1-T50,1,50,Default,1203,304,593,472,190,1129,...,25.475764,20605.846268,18022.772490,20763785.0,Runoff Attenuation Feature,None,None,None,R1-T50.xlsx,
1,OBS-R1-T50,1,50,Storage,2418,842,48,245,554,59,...,24.408278,22336.478805,27272.761678,33463555.0,Runoff Attenuation Feature,None,None,None,OBS-R1-T50.xlsx,
2,OBC-R1-T50,1,50,Carbon,0,340,19,3911,671,108,...,21.142965,88544.371658,62967.032614,20582552.5,Tree Planting,None,None,None,OBC-R1-T50.xlsx,
3,OBH-R1-T50,1,50,Habitat,2418,842,48,245,554,59,...,24.408278,22336.478805,27272.761678,33463555.0,Runoff Attenuation Feature,None,None,None,OBH-R1-T50.xlsx,


In [ ]:
output_name = "ordering_master_summary.xlsx"
summary_df.to_excel(output_name, index=False)

print(f"Saved file: {output_name}")
files.download(output_name)

Saved file: ordering_master_summary.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>